# MSIT — Fine-tune LLaVA-7B / Qwen-VL / InternLM-XComposer2 (QLoRA)
Reproduction of "Open-World Attribute Mining for E-Commerce Products with
Multimodal Self-Correction Instruction Tuning" (ACL 2025).

**Settings (bắt buộc):**
1. Accelerator: GPU T4 x2 hoặc P100 (Settings → Accelerator)
2. Internet: ON (Settings → Internet)
3. Secrets (Add-ons → Secrets): `HF_TOKEN` (để tải model từ HuggingFace),
   `OPENAI_API_KEY` (tùy chọn — chỉ cần nếu muốn dựng AGTD/CTTD mới bằng GPT-4).
4. (Khuyến nghị) Upload dataset AGTD/CTTD + ảnh sản phẩm dạng Kaggle Dataset
   và gắn vào `/kaggle/input/msit-data/`. Nếu không có, notebook tự dựng
   sample nhỏ từ repo để kiểm chử pipeline end-to-end.

Paper hyperparams được giữ nguyên: Adam, lr=3e-4, 10 epochs (Section 4.1).
Khác biệt duy nhất: QLoRA 4-bit thay vì LoRA 16-bit do giới hạn VRAM 16GB.

Training/`LoRA` logic nằm trong package `msit.models` (`LoRATrainer`, backends) —
notebook chỉ orchestrates Kaggle-specific setup, data loading, và gọi package.

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    )
# P100/T4 không hỗ trợ bf16 -> LoRATrainer dùng fp16 (torch_dtype="float16")

In [ ]:
!git clone https://github.com/linhnnh688/msit-open-world.git /kaggle/working/msit-open-world
!cd /kaggle/working/msit-open-world && git log --oneline -3

!pip install -q "transformers==4.44.2" "peft==0.13.2" "accelerate==0.34.2" "bitsandbytes==0.43.3" "datasets==3.0.1" safetensors sentencepiece

import sys
sys.path.insert(0, "/kaggle/working/msit-open-world")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    OPENAI_API_KEY = secrets.get_secret("OPENAI_API_KEY")  # optional
except Exception as e:
    print("Secrets warning:", e)
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

if HF_TOKEN:
    from huggingface_hub import login

    login(token=HF_TOKEN)

from msit.config import MSITConfig

cfg = MSITConfig()

# Env overrides ghi thẳng vào cfg (một nguồn sự thật cho LoRATrainer)
cfg.epochs = int(os.environ.get("EPOCHS", str(cfg.epochs)))  # paper: 10
cfg.n_agtd_samples = int(os.environ.get("MAX_AGTD", str(cfg.n_agtd_samples)))
cfg.n_cttd_samples = int(os.environ.get("MAX_CTTD", str(cfg.n_cttd_samples)))
MAX_AGTD, MAX_CTTD, EPOCHS = cfg.n_agtd_samples, cfg.n_cttd_samples, cfg.epochs

print(
    f"lr={cfg.learning_rate}, epochs={EPOCHS}, MAX_AGTD={MAX_AGTD}, MAX_CTTD={MAX_CTTD}"
)

In [ ]:
import json, os
from msit.data.jsonl import load_jsonl
from msit.models.mock_backend import MockMLLMBackend

DATA_DIR = "/kaggle/input/msit-data"  # Kaggle Dataset do bạn upload (tùy chọn)
agtd_path = os.path.join(DATA_DIR, "agtd.jsonl")
cttd_path = os.path.join(DATA_DIR, "cttd.jsonl")

agtd = load_jsonl(agtd_path)[:MAX_AGTD]
cttd = load_jsonl(cttd_path)[:MAX_CTTD]

if not agtd and not cttd:
    print(
        "[warn] Không tìm thấy AGTD/CTTD trong /kaggle/input/msit-data — "
        "dựng sample nhỏ từ repo để kiểm chứng pipeline."
    )
    with open(
        "/kaggle/working/msit-open-world/msit/data_samples/sample_products.json"
    ) as f:
        prods = json.load(f)["products"]
    db = {p["title"]: p["gold"] for p in prods}
    from msit.data.agtd_builder import AGTDBuilder
    from msit.data.cttd_builder import CTTDBuilder

    agtd = AGTDBuilder(MockMLLMBackend(db)).build_batch(prods)
    cttd = CTTDBuilder(MockMLLMBackend(db)).build_batch(
        [{**p, "attributes": p["gold"]} for p in prods], attributes_per_product=2
    )

train_data = agtd + cttd
print(f"AGTD={len(agtd)}, CTTD={len(cttd)}, total={len(train_data)}")
print(json.dumps(train_data[0], ensure_ascii=False, default=str)[:500])

In [ ]:
import torch
from msit.models.lora_trainer import LoRATrainer

# Kaggle VRAM 16GB: batch nhỏ + accumulation (effective batch ~16)
cfg.per_device_batch_size = 2
cfg.gradient_accumulation_steps = 8


def train_msit_lora(
    model_id: str, vision2seq: bool, output_dir: str, train_data, lora_target_modules
):
    """Thin wrapper — mọi logic QLoRA nằm trong msit.models.lora_trainer.LoRATrainer."""
    trainer = LoRATrainer(
        model_name_or_path=model_id,
        cfg=cfg,
        use_causal_lm=not vision2seq,
        torch_dtype="float16",  # P100/T4 không hỗ trợ bf16
        hf_token=HF_TOKEN or None,
        target_modules=lora_target_modules,
        save_strategy="no",  # chỉ giữ adapter cuối (tiết kiệm disk Kaggle)
    )
    trainer.train(train_data, output_dir)

In [ ]:
# MSIT(LLaVA-7B) — dùng checkpoint llava-hf tương thích transformers
train_msit_lora(
    model_id="llava-hf/llava-1.5-7b-hf",
    vision2seq=True,
    output_dir="/kaggle/working/adapters/msit-llava-7b",
    train_data=train_data,
    lora_target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

In [ ]:
# MSIT(Qwen) — Qwen-VL cần trust_remote_code
train_msit_lora(
    model_id="Qwen/Qwen-VL-Chat",
    vision2seq=False,
    output_dir="/kaggle/working/adapters/msit-qwen-vl-7b",
    train_data=train_data,
    lora_target_modules=["c_attn", "c_proj"],
)

In [ ]:
# MSIT(InternLM)
train_msit_lora(
    model_id="internlm/internlm-xcomposer2-7b",
    vision2seq=False,
    output_dir="/kaggle/working/adapters/msit-internlm-xc2-7b",
    train_data=train_data,
    lora_target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

In [ ]:
!cd /kaggle/working && zip -qr msit_adapters.zip adapters && ls -lh msit_adapters.zip

# Smoke test: load adapter LLaVA qua backend package (apply lora_weights đúng cách)
from msit.models.llava_lora import LLaVALoRABackend
from msit.models.lora_trainer import build_prompt

be = LLaVALoRABackend(
    cfg, lora_weights="/kaggle/working/adapters/msit-llava-7b"
)
be.load()
sample = train_data[0]
print(
    be.generate(
        build_prompt(sample),
        image=sample.get("image"),
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_new_tokens=200,
    )
)